# 03 · Build PyTorch dataset

**Input:** a populated `manifest.csv` and registered glycan JSON/COLVAR/FE index files. **Output:** the `.pt` file at `DATASET_PATH`. This **lean dataset** contains graph tensors, torsion histograms and FE grids; it omits SASA, ring puckering, and conformer-state targets from the original full scientific pipeline.

## Setup

Change `DATABASE_ROOT` and `GLYCAN_ID` in the configuration cell below. Run cells from top to bottom. Dependencies: `numpy`, `pandas`, `PyYAML`, `torch`, `matplotlib`, `networkx`; optional `mdtraj` for PDB inspection. These notebooks are independent: each includes the helper functions it needs.

In [1]:
from __future__ import annotations
import json, math, re, warnings
import yaml
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Optional
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display

In [2]:
DATABASE_ROOT = Path("../data/glycan_database")
GLYCAN_ID = "Man5"
DATASET_PATH = DATABASE_ROOT / "glycan_dataset.pt"

@dataclass
class BuilderConfig:
    database_root: str = str(DATABASE_ROOT)
    manifest_name: str = "manifest.csv"
    output_name: str = "glycan_dataset.pt"
    num_torsion_bins: int = 99
    torsion_min: float = -math.pi
    torsion_max: float = math.pi
    colvar_angle_unit: str = "radian"
    fe_angle_unit: str = "radian"
    sampling_mode: str = "uniform_no_reweighting_required"
    terminal_modifications_last: bool = True
    higher_acceptor_linkage_first: bool = True
    defer_terminal_root_side_branches: bool = True
    yaml_order_fallback: bool = True
    eps: float = 1e-12
    required_preprocessed_xtc_name: str = "preprocessed.xtc"
    glyconformer_auto_generate: bool = False
    glyconformer_output_file: str = "conformer_states.csv"
    glyconformer_separator_file: str = "separator.dat"
    glyconformer_omega_file: str = "omega_angles.dat"

cfg = BuilderConfig()
ROOT = DATABASE_ROOT
MANIFEST = ROOT / cfg.manifest_name
OUTPUT = DATASET_PATH
print("Database:", ROOT.resolve())
print("Dataset output:", OUTPUT.resolve())

Database: /home/niemann/Desktop/test_datatset_builder/data/glycan_database
Dataset output: /home/niemann/Desktop/test_datatset_builder/data/glycan_database/glycan_dataset.pt


In [3]:
TORSION_COL_RE = re.compile(r"^(phi|psi|omega)(\d+)_(\d+)$", re.IGNORECASE)
MANIFEST_COLUMNS = [
    "glycan_id", "structure_json", "colvar_file", "conformer_file",
    "fe_index_file", "trajectory_file", "topology_file", "weight_column"
]

In [4]:
def load_colvar(path):
    """
    Read a whitespace-separated COLVAR with a header:
      first non-comment line = column names
      subsequent lines       = numerical values
    Example columns: time phi1_2 psi1_2 ...
    """
    df = pd.read_csv(path, sep=r"\s+", comment="#")
    if "time" not in df.columns:
        raise ValueError(f"{path}: expected a header containing 'time'")
    return df

In [5]:
LINK_RE = re.compile(r"([abAB])(\d+)-(\d+)")
def parse_linkage(s):
    m = LINK_RE.fullmatch(str(s).strip())
    if not m: return 0.0, 0.0, -1, -1
    a,d,acc = m.groups()
    return float(a.lower()=="a"), float(a.lower()=="b"), int(d), int(acc)

In [6]:
MONO_VOCAB = ["UNK","Glc","Gal","GlcNAc","GalNAc","Neu5Ac","Neu5Gc","Fuc","Man","Xyl","IdoA","GlcA"]
MONO_TO_ID = {x:i for i,x in enumerate(MONO_VOCAB)}
LINKAGE_VOCAB = ["UNK","a1-2","a1-3","a1-4","a1-6","a2-3","a2-6","a2-8","b1-2","b1-3","b1-4","b1-6"]
LINKAGE_TO_ID = {x:i for i,x in enumerate(LINKAGE_VOCAB)}

In [7]:
def load_structure(path):
    """Load a generated structure JSON into root, node lookup, and edge records."""
    o = json.loads(Path(path).read_text())
    nodes = {str(n["id"]): dict(n) for n in o["nodes"]}
    edges = [dict(e, id=str(e["id"]), parent=str(e["parent"]), child=str(e["child"])) for e in o["edges"]]
    return str(o.get("glycan_id", Path(path).stem)), str(o["root_node"]), nodes, edges

In [8]:
def canonicalize(glycan_id, root, nodes, edges):
    """Canonicalize a rooted glycan tree with a chemically meaningful DFS order.

    Primary automatic rule
    ----------------------
    1. Start at the root.
    2. At an ordinary branch point, traverse the child with the highest acceptor
       linkage number first (e.g. 1-6 before 1-3), then backtrack and continue.
    3. If the root has a normal continuing subtree plus one or more terminal
       one-node side branches, defer those root-attached leaves until the very
       end of the whole traversal. This makes a core Fuc attached to the first
       GlcNAc the final canonical node without hard-coding the monosaccharide.

    Safety backup
    -------------
    register_glycan() stores the verified insertion order from glycan.yaml as
    node['yaml_position']. If the automatic order disagrees with that complete
    YAML order and cfg.yaml_order_fallback is True, YAML order wins with a warning.
    """
    children = defaultdict(list)
    indeg = Counter()
    incoming = {}
    for e in edges:
        if e["parent"] not in nodes or e["child"] not in nodes:
            raise ValueError(f"{glycan_id}: edge {e['id']} references unknown node")
        children[e["parent"]].append(e)
        indeg[e["child"]] += 1
        incoming[e["child"]] = e

    if root not in nodes or indeg[root] != 0:
        raise ValueError(f"{glycan_id}: invalid root {root}")
    for nid in nodes:
        if nid != root and indeg[nid] != 1:
            raise ValueError(f"{glycan_id}: expected rooted acyclic tree; node {nid} indegree={indeg[nid]}")

    def subtree_signature(nid):
        parts = []
        for e in children.get(nid, []):
            parts.append((e["linkage"], nodes[e["child"]]["mono"], subtree_signature(e["child"])))
        return (nodes[nid]["mono"], tuple(sorted(parts, reverse=True)))

    def sort_key(e):
        child = nodes[e["child"]]
        alpha,beta,donor,acceptor = parse_linkage(e["linkage"])
        terminal = int(bool(child.get("terminal_modification", False))) if cfg.terminal_modifications_last else 0
        acc_rank = -acceptor if cfg.higher_acceptor_linkage_first else acceptor
        anomer_rank = 0 if alpha else (1 if beta else 2)
        return (terminal, acc_rank, -donor, anomer_rank,
                MONO_TO_ID.get(child["mono"],9999), child["mono"],
                repr(subtree_signature(e["child"])), e["id"])

    # Root-attached terminal leaves are a special *structural* case. We defer
    # them only when the root also has at least one continuing (non-leaf) child.
    # Thus no monosaccharide identity (Fuc or otherwise) is guessed.
    root_edges = list(children.get(root, []))
    deferred_root_edges = []
    if cfg.defer_terminal_root_side_branches and len(root_edges) > 1:
        continuing = [e for e in root_edges if len(children.get(e["child"], [])) > 0]
        if continuing:
            deferred_root_edges = [e for e in root_edges if len(children.get(e["child"], [])) == 0]

    auto_order, visited = [], set()
    def dfs(nid):
        if nid in visited:
            raise ValueError(f"{glycan_id}: cycle detected")
        visited.add(nid)
        auto_order.append(nid)
        es = list(children.get(nid, []))
        if nid == root and deferred_root_edges:
            deferred_ids = {e["id"] for e in deferred_root_edges}
            es = [e for e in es if e["id"] not in deferred_ids]
        for e in sorted(es, key=sort_key):
            dfs(e["child"])

    dfs(root)
    for e in sorted(deferred_root_edges, key=sort_key):
        dfs(e["child"])

    if len(visited) != len(nodes):
        raise ValueError(f"{glycan_id}: disconnected nodes {sorted(set(nodes)-visited)}")

    # Complete YAML order is available only for structures generated by v7.5+.
    yaml_positions = {nid: nodes[nid].get("yaml_position") for nid in nodes}
    have_yaml_order = all(isinstance(v, int) for v in yaml_positions.values())
    yaml_order = (sorted(nodes, key=lambda nid: yaml_positions[nid]) if have_yaml_order else None)

    node_order = list(auto_order)
    order_source = "automatic_graph_rule"
    if yaml_order is not None and auto_order != yaml_order:
        msg = (f"{glycan_id}: automatic canonical node order {auto_order} disagrees with "
               f"verified glycan.yaml order {yaml_order}.")
        if cfg.yaml_order_fallback:
            warnings.warn(msg + " Using glycan.yaml order as safety fallback.")
            node_order = list(yaml_order)
            order_source = "yaml_fallback_after_graph_disagreement"
        else:
            raise ValueError(msg + " YAML fallback is disabled.")
    elif yaml_order is not None:
        order_source = "automatic_graph_rule_verified_by_yaml"

    # In a rooted tree every non-root node has exactly one incoming edge. Ordering
    # physical edges by the canonical position of their child guarantees that edge
    # and torsion numbering stay synchronized with canonical node numbering.
    edge_order = [incoming[nid]["id"] for nid in node_order if nid != root]
    edge_by_id = {e["id"]: e for e in edges}

    return {
        "node_order": node_order,
        "automatic_node_order": auto_order,
        "yaml_node_order": yaml_order,
        "order_source": order_source,
        "edge_order": edge_order,
        "node_to_idx": {n:i for i,n in enumerate(node_order)},
        "edge_to_idx": {e:i for i,e in enumerate(edge_order)},
        "edges": [edge_by_id[e] for e in edge_order],
    }

In [9]:
def canonical_dihedral_index(fe_index_path,canon):
    """Sort FE-index rows by canonical edge order and create canonical dihedral definitions."""
    df=pd.read_csv(fe_index_path)
    req={"edge_id","torsion_name","colvar_column","fe_file"}
    if req-set(df.columns): raise ValueError(f"Missing FE-index columns: {sorted(req-set(df.columns))}")
    df=df.copy(); df["edge_id"]=df["edge_id"].astype(str)
    unknown=set(df.edge_id)-set(canon["edge_to_idx"])
    if unknown: raise ValueError(f"Unknown edge IDs in FE index: {sorted(unknown)}")
    df["_edge"]=df.edge_id.map(canon["edge_to_idx"])
    df["_within"]=df.groupby("edge_id").cumcount()
    df=df.sort_values(["_edge","_within"],kind="stable").reset_index(drop=True)
    defs=[]
    for di,row in df.iterrows():
        e=canon["edges"][int(row["_edge"])]
        defs.append({
            "dihedral_id":int(di),"physical_edge_id":int(row["_edge"]),
            "raw_edge_id":e["id"],"source_node":canon["node_to_idx"][e["parent"]],
            "target_node":canon["node_to_idx"][e["child"]],"linkage":e["linkage"],
            "torsion_name":str(row["torsion_name"]),"colvar_column":str(row["colvar_column"]),
            "fe_file":str(row["fe_file"])
        })
    return df,defs

In [10]:
def prepare_manifest(manifest_path=MANIFEST):
    m = pd.read_csv(manifest_path)
    req = {"glycan_id", "structure_json", "colvar_file", "conformer_file", "fe_index_file"}
    missing = req - set(m.columns)
    if missing:
        raise ValueError(f"Manifest missing {sorted(missing)}")
    if m.glycan_id.astype(str).duplicated().any():
        raise ValueError("Duplicate glycan_id in manifest")
    return m

In [11]:
def resolve(base, value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    s = str(value).strip()
    if not s or s.lower() == "nan":
        return None
    p = Path(s)
    return p if p.is_absolute() else Path(base) / p

In [12]:
def mono_charge(m):
    return -1.0 if m in {"Neu5Ac","Neu5Gc","GlcA","IdoA"} else 0.0
def mono_carboxyl(m):
    return float(m in {"Neu5Ac","Neu5Gc","GlcA","IdoA"})
def mono_nacetyl(m):
    return float(m in {"GlcNAc","GalNAc","Neu5Ac"})
def mono_sulfate(m):
    return 0.0  # extend vocabulary/descriptor when sulfated residues are added

In [13]:
def graph_arrays(nodes, canon):
    M = len(canon["node_order"])
    degree=np.zeros(M,np.float32); indeg=np.zeros(M,np.float32); outdeg=np.zeros(M,np.float32)
    src=[]; dst=[]; linkage_ids=[]; edge_float=[]; physical_edge_id=[]
    physical_linkage_ids=[]; physical_edge_float=[]

    for ei,e in enumerate(canon["edges"]):
        u=canon["node_to_idx"][e["parent"]]; v=canon["node_to_idx"][e["child"]]
        degree[u]+=1; degree[v]+=1; outdeg[u]+=1; indeg[v]+=1
        a,b,d,acc=parse_linkage(e["linkage"])
        base=np.array([a,b,d/9 if d>=0 else 0,acc/9 if acc>=0 else 0],np.float32)
        lid=LINKAGE_TO_ID.get(e["linkage"],0)
        physical_linkage_ids.append(lid); physical_edge_float.append(base)
        for s,t,direction in [(u,v,1.0),(v,u,-1.0)]:
            src.append(s); dst.append(t); linkage_ids.append(lid)
            edge_float.append(np.r_[base,direction].astype(np.float32)); physical_edge_id.append(ei)

    node_float=[]; mono_ids=[]
    for i,nid in enumerate(canon["node_order"]):
        n=nodes[nid]; m=n["mono"]; mono_ids.append(MONO_TO_ID.get(m,0))
        node_float.append([
            mono_carboxyl(m), mono_charge(m), mono_sulfate(m), mono_nacetyl(m),
            degree[i]/4, indeg[i]/4, outdeg[i]/4,
            float(degree[i]<=1), float(outdeg[i]>1)
        ])

    return {
        "mono_ids":torch.tensor(mono_ids,dtype=torch.long),
        "node_float":torch.tensor(np.asarray(node_float),dtype=torch.float32),
        "edge_index":torch.tensor([src,dst],dtype=torch.long),
        "linkage_ids":torch.tensor(linkage_ids,dtype=torch.long),
        "edge_float":torch.tensor(np.asarray(edge_float),dtype=torch.float32),
        "physical_edge_id":torch.tensor(physical_edge_id,dtype=torch.long),
        "physical_linkage_ids":torch.tensor(physical_linkage_ids,dtype=torch.long),
        "physical_edge_float":torch.tensor(np.asarray(physical_edge_float),dtype=torch.float32),
    }

In [14]:
def _angle_radians(values, unit):
    """Convert angle arrays to radians; reject unknown angle units."""
    values = np.asarray(values, dtype=float)
    if unit == "radian":
        return values
    if unit == "degree":
        return np.deg2rad(values)
    raise ValueError(f"Unknown angle unit: {unit}")


def _torsion_grid():
    """Return periodic histogram edges and bin centers in radians."""
    edges = np.linspace(cfg.torsion_min, cfg.torsion_max, cfg.num_torsion_bins + 1)
    return edges, (edges[:-1] + edges[1:]) / 2


def _histogram(values):
    """Create a normalized periodic torsion histogram using uniform frame weights."""
    edges, _ = _torsion_grid()
    angles = _angle_radians(values, cfg.colvar_angle_unit)
    angles = (angles - edges[0]) % (2 * np.pi) + edges[0]
    hist, _ = np.histogram(angles, bins=edges)
    hist = hist.astype(float) + cfg.eps
    return torch.tensor(hist / hist.sum(), dtype=torch.float32)


def _fe_on_grid(path):
    """Read a two-column FE profile and interpolate it periodically onto torsion bin centers.

    Assumes column 1 is angle and column 2 is FE (same energy units across glycans).
    """
    data = np.loadtxt(path, comments="#", usecols=(0, 1), ndmin=2)
    if data.shape[0] < 2:
        raise ValueError(f"FE profile requires >= 2 samples: {path}")
    x = _angle_radians(data[:, 0], cfg.fe_angle_unit)
    y = np.asarray(data[:, 1], dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if len(x) < 2:
        raise ValueError(f"Not enough finite FE samples: {path}")
    x = (x + np.pi) % (2 * np.pi) - np.pi
    order = np.argsort(x)
    x, y = x[order], y[order]
    x, unique_idx = np.unique(x, return_index=True)
    y = y[unique_idx]
    if len(x) < 2:
        raise ValueError(f"FE angles collapse to one unique point: {path}")
    _, centers = _torsion_grid()
    interpolated = np.interp(centers, np.r_[x[-1]-2*np.pi, x, x[0]+2*np.pi],
                             np.r_[y[-1], y, y[0]])
    interpolated -= interpolated.min()
    return torch.tensor(interpolated, dtype=torch.float32)


def build_one_glycan(row):
    """Convert one manifest entry into canonical graph and torsion tensors."""
    gid = str(row["glycan_id"])
    structure_path = resolve(ROOT, row["structure_json"])
    colvar_path = resolve(ROOT, row["colvar_file"])
    fe_index_path = resolve(ROOT, row["fe_index_file"])
    _, root, nodes, edges = load_structure(structure_path)
    canon = canonicalize(gid, root, nodes, edges)
    graph = graph_arrays(nodes, canon)
    _, defs = canonical_dihedral_index(fe_index_path, canon)
    colvar = load_colvar(colvar_path)
    histograms, energies = [], []
    for definition in defs:
        name = definition["colvar_column"]
        if name not in colvar.columns:
            raise ValueError(f"{gid}: missing COLVAR column {name}")
        histograms.append(_histogram(colvar[name].to_numpy(float)))
        energies.append(_fe_on_grid(resolve(fe_index_path.parent, definition["fe_file"])))
    if not histograms:
        raise ValueError(f"{gid}: no torsions found")
    record = {
        "glycan_id": gid,
        **graph,
        "canonical_nodes": canon["node_order"],
        "dihedral_definitions": defs,
        "torsion_histograms": torch.stack(histograms),
        "torsion_free_energies": torch.stack(energies),
        "n_frames": len(colvar),
        "conformer_states": None,
    }
    conformer_path = resolve(ROOT, row.get("conformer_file"))
    if conformer_path is not None and conformer_path.is_file():
        states = load_conformer_states(conformer_path)
        validate_conformer_dihedral_order(states, canonical_dihedral_names(defs), gid)
        if len(states) != len(colvar):
            raise ValueError(f"{gid}: conformer/COLVAR frame counts differ")
        record["conformer_states"] = states.to_dict(orient="list")
    return record


def build_dataset(manifest_path=MANIFEST, output_path=OUTPUT):
    """Build all registered glycans and save a combined PyTorch .pt file."""
    manifest = prepare_manifest(manifest_path)
    if manifest.empty:
        raise ValueError("Manifest has no registered glycans")
    records = [build_one_glycan(row) for _, row in manifest.iterrows()]
    edges, centers = _torsion_grid()
    dataset = {
        "format_version": "glycan_registration_simple_v1",
        "glycans": records,
        "mono_vocab": MONO_VOCAB,
        "linkage_vocab": LINKAGE_VOCAB,
        "torsion_grid_edges": torch.tensor(edges, dtype=torch.float32),
        "torsion_grid_centers": torch.tensor(centers, dtype=torch.float32),
        "settings": {"colvar_angle_unit": cfg.colvar_angle_unit,
                     "fe_angle_unit": cfg.fe_angle_unit,
                     "fe_units": "unchanged from input FE profiles",
                     "frame_weights": "uniform"},
    }
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(dataset, output_path)
    print(f"Saved {len(records)} glycans to {output_path.resolve()}")
    return dataset

## Run

Build the dataset from **all** registered glycans and save it to disk. The build function uses the manifest, not `GLYCAN_ID`.

In [15]:
dataset = build_dataset()
print("Saved:", DATASET_PATH)
print("Glycans:", len(dataset["glycans"]))

Saved 1 glycans to /home/niemann/Desktop/test_datatset_builder/data/glycan_database/glycan_dataset.pt
Saved: ../data/glycan_database/glycan_dataset.pt
Glycans: 1
